# NC Capstone: A.I. Business Sentiment and Employment by State
This notebook loads and combines BLS and BTOS datasets using relative paths, then seeks to analyze the relationship between A.I. sentiment and employment by type and U.S. State over time. 

---

Load Libraries

In [1]:
import pandas as pd
from glob import glob
import os

Checking to Make Sure File Path is Working

In [2]:
bls_paths = sorted(glob("data/state_M20*_dl.xlsx"))
print("Files found:")
print(bls_paths)

Files found:
['data\\state_M2020_dl.xlsx', 'data\\state_M2021_dl.xlsx', 'data\\state_M2022_dl.xlsx', 'data\\state_M2023_dl.xlsx', 'data\\state_M2024_dl.xlsx']


Load and Combine BLS Data
This block loads all BLS Excel files from the `data/` folder.

In [3]:

bls_dfs = []

for path in bls_paths:
    try:
        df = pd.read_excel(path)
        df["source_file"] = os.path.basename(path)
        bls_dfs.append(df)
    except Exception as e:
        print(f"Failed to load {path}: {e}")

# Combine all BLS data
bls_combined = pd.concat(bls_dfs, ignore_index=True)
print("\n BLS data combined. Sample:")
display(bls_combined.head())


 BLS data combined. Sample:


,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY,source_file,PCT_RPT
0,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,41.07,18690,24060,36250,56980,85430,NaN,NaN,state_M2020_dl.xlsx,NaN
1,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,91.89,47740,67330,95120,134320,191130,NaN,NaN,state_M2020_dl.xlsx,NaN
2,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,#,49480,97930,161290,#,#,NaN,NaN,state_M2020_dl.xlsx,NaN
3,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,#,48030,67740,101170,153050,#,NaN,NaN,state_M2020_dl.xlsx,NaN
4,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1031,Legislators,...,*,16220,17190,18820,27920,55970,True,NaN,state_M2020_dl.xlsx,NaN


Adding 'Tech' Classification to BLS Occupational Data

In [45]:
# Add tech classification column to BLS data
print("Adding tech classification to BLS data")

# Create the tech flag based on OCC_TITLE containing "Data" or "Software"
bls_combined['Tech_Classification'] = bls_combined['OCC_TITLE'].apply(
    lambda x: 'Tech' if pd.notna(x) and ('DATA' in str(x).upper() or 'SOFTWARE' in str(x).upper()) 
    else 'Non Tech'
)

# Verify the classification
tech_counts = bls_combined['Tech_Classification'].value_counts()
print(f"\nTech Classification Counts:")
print(tech_counts)

# Show some examples of Tech-classified jobs
print(f"\nSample Tech-classified occupations:")
tech_jobs = bls_combined[bls_combined['Tech_Classification'] == 'Tech']['OCC_TITLE'].unique()
print(tech_jobs[:10])  # Show first 10 unique tech job titles

print(f"\nBLS data updated with Tech_Classification column.")
print(f"New shape: {bls_combined.shape}")

Adding tech classification to BLS data

Tech Classification Counts:
Tech_Classification
Non Tech    184322
Tech          1432
Name: count, dtype: int64

Sample Tech-classified occupations:
['Database Administrators and Architects'
 'Software Developers and Software Quality Assurance Analysts and Testers'
 'Data Scientists and Mathematical Science Occupations, All Other'
 'Data Entry Keyers' 'Database Administrators' 'Database Architects'
 'Software Developers' 'Software Quality Assurance Analysts and Testers'
 'Data Scientists']

BLS data updated with Tech_Classification column.
New shape: (185754, 34)


BTOS Collection Date Translation Added Here

In [46]:
date_translation_path = "data/btos_collection_dates.csv"
try:
    date_translation = pd.read_csv(date_translation_path)
    print("BTOS date translation loaded successfully.")
    display(date_translation.head())
except Exception as e:
    print(f"Failed to load {date_translation_path}: {e}")

BTOS date translation loaded successfully.


,Smpdt,Col Start,Col End,Ref Start,Ref End
0,202215,07/18/2022,07/31/2022,07/04/2022,07/17/2022
1,202216,08/01/2022,08/14/2022,07/18/2022,07/31/2022
2,202217,08/15/2022,08/28/2022,08/01/2022,08/14/2022
3,202218,08/29/2022,09/11/2022,08/15/2022,08/28/2022
4,202219,09/12/2022,09/25/2022,08/29/2022,09/11/2022


Load and Combine BTOS Data.  
This block loads the BTOS survey data files from the `data/` folder.

In [47]:
btos_paths = ["data/State.xlsx", "data/State_v1.xlsx"]
btos_dfs = []

for path in btos_paths:
    try:
        df = pd.read_excel(path)
        df["source_file"] = os.path.basename(path)
        btos_dfs.append(df)
    except Exception as e:
        print(f"Failed to load {path}: {e}")

# Combine all BTOS data
btos_combined = pd.concat(btos_dfs, ignore_index=True)
print("\n BTOS data combined. Sample:")
display(btos_combined.head())


 BTOS data combined. Sample:


,State,Question ID,Question,Answer ID,Answer,202512,202511,202510,202509,202508,...,202224,202223,202222,202221,202220,202219,202218,202217,202216,202215
0,AK,2.0,"Overall, how would you describe this business'...",1.0,Excellent,S,S,S,S,S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AK,2.0,"Overall, how would you describe this business'...",2.0,Above average,17.9%,16.5%,13.9%,19.6%,21.4%,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AK,2.0,"Overall, how would you describe this business'...",3.0,Average,46.2%,52.9%,69.6%,34.6%,57.1%,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AK,2.0,"Overall, how would you describe this business'...",4.0,Below average,18.2%,18.2%,S,32.1%,S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AK,2.0,"Overall, how would you describe this business'...",5.0,Poor,S,S,S,S,S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Renaming BTOS Date Columns to Make them More Usable

In [48]:

collection_dates = pd.read_csv(date_translation_path)
print("BTOS collection dates loaded successfully.")
display(collection_dates.head())

# Convert Smpdt to string and create mapping
collection_dates['Smpdt'] = collection_dates['Smpdt'].astype(str)
date_mapping = dict(zip(collection_dates['Smpdt'], collection_dates['Ref End']))

# Check current BTOS columns
print("Current BTOS date columns:", [col for col in btos_combined.columns if str(col).startswith('202')][:10])

# Find matching columns and rename
date_columns = [col for col in btos_combined.columns if str(col) in date_mapping.keys()]
print(f"Found {len(date_columns)} matching columns to rename")

if len(date_columns) > 0:
    rename_dict = {col: date_mapping[str(col)] for col in date_columns}
    btos_combined = btos_combined.rename(columns=rename_dict)
    print(f"Successfully renamed {len(rename_dict)} date columns to Ref End dates.")
    
    # Verify the renaming worked
    print("Sample new column names:", list(btos_combined.columns)[-10:])
else:
    print("No matching columns found")

BTOS collection dates loaded successfully.


,Smpdt,Col Start,Col End,Ref Start,Ref End
0,202215,07/18/2022,07/31/2022,07/04/2022,07/17/2022
1,202216,08/01/2022,08/14/2022,07/18/2022,07/31/2022
2,202217,08/15/2022,08/28/2022,08/01/2022,08/14/2022
3,202218,08/29/2022,09/11/2022,08/15/2022,08/28/2022
4,202219,09/12/2022,09/25/2022,08/29/2022,09/11/2022


Current BTOS date columns: ['202512', '202511', '202510', '202509', '202508', '202507', '202506', '202505', '202504', '202503']
Found 76 matching columns to rename
Successfully renamed 76 date columns to Ref End dates.
Sample new column names: ['11/20/2022', '11/06/2022', '10/23/2022', '10/09/2022', '09/25/2022', '09/11/2022', '08/28/2022', '08/14/2022', '07/31/2022', '07/17/2022']


Adding 'Region' Column to Both Files

In [52]:
region_mapping = {
    # Northeast
    'ME': 'Northeast', 'NH': 'Northeast', 'VT': 'Northeast', 'MA': 'Northeast', 
    'RI': 'Northeast', 'CT': 'Northeast', 'NY': 'Northeast', 'PA': 'Northeast', 
    'NJ': 'Northeast', 'DE': 'Northeast', 'MD': 'Northeast',
    
    # Southeast
    'FL': 'Southeast', 'AL': 'Southeast', 'GA': 'Southeast', 'SC': 'Southeast', 
    'NC': 'Southeast', 'VA': 'Southeast', 'WV': 'Southeast', 'KY': 'Southeast', 
    'TN': 'Southeast', 'MS': 'Southeast', 'AR': 'Southeast', 'LA': 'Southeast',
    
    # Midwest
    'OH': 'Midwest', 'IN': 'Midwest', 'MI': 'Midwest', 'IL': 'Midwest', 
    'WI': 'Midwest', 'MN': 'Midwest', 'IA': 'Midwest', 'MO': 'Midwest', 
    'ND': 'Midwest', 'SD': 'Midwest', 'NE': 'Midwest', 'KS': 'Midwest',
    
    # Southwest
    'TX': 'Southwest', 'NM': 'Southwest', 'AZ': 'Southwest', 'OK': 'Southwest',
    
    # West
    'CA': 'West', 'OR': 'West', 'WA': 'West', 'NV': 'West', 'ID': 'West', 
    'MT': 'West', 'WY': 'West', 'UT': 'West', 'CO': 'West', 'AK': 'West', 'HI': 'West',

    # Territories
    'DC': 'Territories', 'GU': 'Territories', 'PR': 'Territories', 'VI': 'Territories',
    'XX': 'Territories'
}

# Add Region column to BLS data
print("Adding regional classification to BLS data...")
bls_combined['Region'] = bls_combined['PRIM_STATE'].map(region_mapping)

# Add Region column to BTOS data  
print("Adding regional classification to BTOS data...")
btos_combined['Region'] = btos_combined['State'].map(region_mapping)

# Verify the regional classifications
print(f"\nBLS Regional Distribution:")
print(bls_combined['Region'].value_counts())

print(f"\nBTOS Regional Distribution:")
print(btos_combined['Region'].value_counts())

# Check for any unmapped states
bls_unmapped = bls_combined[bls_combined['Region'].isna()]['PRIM_STATE'].unique()
btos_unmapped = btos_combined[btos_combined['Region'].isna()]['State'].unique()

if len(bls_unmapped) > 0:
    print(f"\nUnmapped BLS states: {bls_unmapped}")
if len(btos_unmapped) > 0:
    print(f"\nUnmapped BTOS states: {btos_unmapped}")

print(f"\nRegional classification complete!")
print(f"BLS data shape: {bls_combined.shape}")
print(f"BTOS data shape: {btos_combined.shape}")

Adding regional classification to BLS data...
Adding regional classification to BTOS data...

BLS Regional Distribution:
Region
Southeast      44633
Midwest        43745
Northeast      37951
West           36975
Southwest      14799
Territories     7651
Name: count, dtype: int64

BTOS Regional Distribution:
Region
Southeast      2160
Midwest        2160
West           1980
Northeast      1980
Southwest       720
Territories     455
Name: count, dtype: int64

Unmapped BTOS states: [nan
 'Source: U.S. Census Bureau, Business Trends and Outlook Survey (BTOS). The Census Bureau has reviewed this data product to ensure appropriate access, use, and disclosure avoidance protection of the confidential source data.\n        (Project No. P-7529868, Disclosure Review Board (DRB) approval number: CBDRB-FY24-0474)'
 'Source: U.S. Census Bureau, Business Trends and Outlook Survey (BTOS), Posted date: September 14, 2023, Project No. EID-7529868 / Approval CBDRB-FY22-341.']

Regional classification co

Save Combined Outputs

In [49]:
bls_combined.to_csv("data/combined_bls.csv", index=False)
btos_combined.to_csv("data/combined_btos.csv", index=False)
print("\n Output saved: data/combined_bls.csv and data/combined_btos.csv")


 Output saved: data/combined_bls.csv and data/combined_btos.csv


Combined Staging Data Preview

In [50]:
print("\nPreview: BLS Combined Data")
display(bls_combined.head())

print("\nPreview: BTOS Combined Data")
display(btos_combined.head())


Preview: BLS Combined Data


,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY,source_file,PCT_RPT,Tech_Classification
0,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,18690,24060,36250,56980,85430,NaN,NaN,state_M2020_dl.xlsx,NaN,Non Tech
1,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,47740,67330,95120,134320,191130,NaN,NaN,state_M2020_dl.xlsx,NaN,Non Tech
2,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,49480,97930,161290,#,#,NaN,NaN,state_M2020_dl.xlsx,NaN,Non Tech
3,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,48030,67740,101170,153050,#,NaN,NaN,state_M2020_dl.xlsx,NaN,Non Tech
4,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1031,Legislators,...,16220,17190,18820,27920,55970,True,NaN,state_M2020_dl.xlsx,NaN,Non Tech



Preview: BTOS Combined Data


,State,Question ID,Question,Answer ID,Answer,06/01/2025,05/18/2025,05/04/2025,04/20/2025,04/06/2025,...,11/20/2022,11/06/2022,10/23/2022,10/09/2022,09/25/2022,09/11/2022,08/28/2022,08/14/2022,07/31/2022,07/17/2022
0,AK,2.0,"Overall, how would you describe this business'...",1.0,Excellent,S,S,S,S,S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AK,2.0,"Overall, how would you describe this business'...",2.0,Above average,17.9%,16.5%,13.9%,19.6%,21.4%,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AK,2.0,"Overall, how would you describe this business'...",3.0,Average,46.2%,52.9%,69.6%,34.6%,57.1%,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AK,2.0,"Overall, how would you describe this business'...",4.0,Below average,18.2%,18.2%,S,32.1%,S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AK,2.0,"Overall, how would you describe this business'...",5.0,Poor,S,S,S,S,S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Selecting the Relevant A.I. Questions from the BTOS Survey

In [42]:
from pandasql import sqldf

btos_combined.columns = btos_combined.columns.str.replace(' ', '_')
btos_combined = btos_combined.loc[:, ~btos_combined.columns.isna()]

# I am only using the "State" source - which goes as far back as Sept 2023, 
# (cont'd) because prior to Sept 2023 (State v1 file) there was not an A.I. question in the data.

query = """
SELECT distinct Question, Question_ID
FROM btos_combined
WHERE 1=1
and Question like '%Artificial Intelligence%'
and source_file in ('State.xlsx')
limit 10
"""
print("\nQuerying BTOS Combined Data for AI-related questions:")


result = sqldf(query, locals())


print(result)


Querying BTOS Combined Data for AI-related questions:
                                            Question  Question_ID
0  In the last two weeks, did this business use A...          7.0
1  During the next six months, do you think this ...         26.0


Getting Cleaned BTOS Dataset Together

In [44]:
# Filter BTOS data to only AI-related questions
print("Filtering BTOS data to AI-related questions only")

# Filter for only the two AI questions we identified
ai_questions_filter = (btos_combined['Question_ID'].isin([7.0, 26.0])) & \
                     (btos_combined['source_file'] == 'State.xlsx')

btos_ai_only = btos_combined[ai_questions_filter].copy()

print(f"Original BTOS data shape: {btos_combined.shape}")
print(f"Filtered AI-only data shape: {btos_ai_only.shape}")
print(f"Reduction: {btos_combined.shape[0] - btos_ai_only.shape[0]:,} rows removed")

# Verify the filtering worked
print("\nUnique questions remaining:")
verification_query = """
SELECT DISTINCT Question_ID, Question, COUNT(*) as row_count
FROM btos_ai_only
GROUP BY Question_ID, Question
"""
verification_result = sqldf(verification_query, locals())
print(verification_result)

# Update the main dataframe reference
btos_combined = btos_ai_only
print("\nBTOS dataset successfully reduced to AI-related questions only.")

Filtering BTOS data to AI-related questions only
Original BTOS data shape: (318, 52)
Filtered AI-only data shape: (318, 52)
Reduction: 0 rows removed

Unique questions remaining:
   Question_ID                                           Question  row_count
0          7.0  In the last two weeks, did this business use A...        159
1         26.0  During the next six months, do you think this ...        159

BTOS dataset successfully reduced to AI-related questions only.


Cleaning BLS Data